# Sklearn patch-wise Recipe

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jejjohnson/xr_toolz/blob/main/docs/notebooks/sklearn_patchwise_recipe.ipynb)

This recipe shows how `XarrayEstimator` composes with patch-wise xarray workflows: fit local sklearn estimators independently on each spatial patch, or fit one global estimator and run memory-bounded inference on patches that preserve the fitted feature axis.

**What you'll learn:**

1. How per-patch fitting works when each spatial tile has its own local statistics.
2. Why global-fit / patch-infer workflows must patch along the sample axis so every patch keeps the same feature layout seen during fitting.
3. Where `XRDAPatcher` slots into the loop without any additional xr_toolz operator.

In [1]:
import subprocess
import sys


try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "xr_toolz @ git+https://github.com/jejjohnson/xr_toolz@main",
        ],
        check=True,
    )

In [2]:
import importlib.util
import warnings

import numpy as np
import xarray as xr
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

from xr_toolz.utils import XarrayEstimator


warnings.filterwarnings("ignore", message=r".*IProgress.*")

try:
    from IPython import get_ipython

    ipython = get_ipython()
except ImportError:
    ipython = None

if ipython is not None and importlib.util.find_spec("watermark") is not None:
    ipython.run_line_magic("load_ext", "watermark")
    ipython.run_line_magic("watermark", "-v -m -p numpy,xarray,sklearn,xr_toolz")
else:
    print("watermark extension not installed; skipping reproducibility readout.")

Python implementation: CPython
Python version       : 3.12.3
IPython version      : 9.10.0

numpy   : 2.4.4
xarray  : 2026.4.0
sklearn : 1.8.0
xr_toolz: 0.0.5

Compiler    : GCC 13.3.0
OS          : Linux
Release     : 6.17.0-1010-azure
Machine     : x86_64
Processor   : x86_64
CPU cores   : 4
Architecture: 64bit



## Synthetic sea-surface-height cube

The examples use a tiny `time × lat × lon` `DataArray` so the patch shapes are visible. Real workflows replace the hand-written `isel` slices below with `XRDAPatcher(da=ssh, patches=..., strides=...)`.

In [3]:
time = np.arange(6)
lat = np.linspace(-30.0, 30.0, 4)
lon = np.linspace(140.0, 143.0, 4)
values = (
    np.sin(time[:, None, None] / 2.0)
    + lat[None, :, None] / 100.0
    + lon[None, None, :] / 1000.0
)
ssh = xr.DataArray(
    values,
    dims=("time", "lat", "lon"),
    coords={"time": time, "lat": lat, "lon": lon},
    name="ssh",
    attrs={"units": "m"},
)
ssh

<xarray.DataArray 'ssh' (time: 6, lat: 4, lon: 4)> Size: 768B
array([[[-0.16      , -0.159     , -0.158     , -0.157     ],
        [ 0.04      ,  0.041     ,  0.042     ,  0.043     ],
        [ 0.24      ,  0.241     ,  0.242     ,  0.243     ],
        [ 0.44      ,  0.441     ,  0.442     ,  0.443     ]],

       [[ 0.31942554,  0.32042554,  0.32142554,  0.32242554],
        [ 0.51942554,  0.52042554,  0.52142554,  0.52242554],
        [ 0.71942554,  0.72042554,  0.72142554,  0.72242554],
        [ 0.91942554,  0.92042554,  0.92142554,  0.92242554]],

       [[ 0.68147098,  0.68247098,  0.68347098,  0.68447098],
        [ 0.88147098,  0.88247098,  0.88347098,  0.88447098],
        [ 1.08147098,  1.08247098,  1.08347098,  1.08447098],
        [ 1.28147098,  1.28247098,  1.28347098,  1.28447098]],

       [[ 0.83749499,  0.83849499,  0.83949499,  0.84049499],
        [ 1.03749499,  1.03849499,  1.03949499,  1.04049499],
        [ 1.23749499,  1.23849499,  1.23949499,  1.24049499],
        [ 1.43749499,  1.43849499,  1.43949499,  1.44049499]],

       [[ 0.74929743,  0.75029743,  0.75129743,  0.75229743],
        [ 0.94929743,  0.95029743,  0.95129743,  0.95229743],
        [ 1.14929743,  1.15029743,  1.15129743,  1.15229743],
        [ 1.34929743,  1.35029743,  1.35129743,  1.35229743]],

       [[ 0.43847214,  0.43947214,  0.44047214,  0.44147214],
        [ 0.63847214,  0.63947214,  0.64047214,  0.64147214],
        [ 0.83847214,  0.83947214,  0.84047214,  0.84147214],
        [ 1.03847214,  1.03947214,  1.04047214,  1.04147214]]])
Coordinates:
  * time     (time) int64 48B 0 1 2 3 4 5
  * lat      (lat) float64 32B -30.0 -10.0 10.0 30.0
  * lon      (lon) float64 32B 140.0 141.0 142.0 143.0
Attributes:
    units:    m

## Pattern 1 — per-patch local fit

Use this pattern when each patch should learn its own local statistics, such as local scaling before a regional EOF/PCA. With `XRDAPatcher`, the loop shape is:

```python
from xrpatcher import XRDAPatcher

patcher = XRDAPatcher(da=ssh, patches={"lat": 64, "lon": 64}, strides={"lat": 64, "lon": 64})
scaled = [
    XarrayEstimator(StandardScaler(), sample_dim="time").fit_transform(patcher[i])
    for i in range(len(patcher))
]
out = patcher.reconstruct(scaled)
```

In [4]:
spatial_slices = [
    {"lat": slice(0, 2), "lon": slice(0, 2)},
    {"lat": slice(0, 2), "lon": slice(2, 4)},
    {"lat": slice(2, 4), "lon": slice(0, 2)},
    {"lat": slice(2, 4), "lon": slice(2, 4)},
]
local_scaled_patches = [
    XarrayEstimator(StandardScaler(), sample_dim="time").fit_transform(ssh.isel(**slc))
    for slc in spatial_slices
]
local_scaled = xr.combine_by_coords(local_scaled_patches)["ssh"]
local_scaled

<xarray.DataArray 'ssh' (time: 6, lat: 4, lon: 4)> Size: 768B
array([[[-1.8970362 , -1.8970362 , -1.8970362 , -1.8970362 ],
        [-1.8970362 , -1.8970362 , -1.8970362 , -1.8970362 ],
        [-1.8970362 , -1.8970362 , -1.8970362 , -1.8970362 ],
        [-1.8970362 , -1.8970362 , -1.8970362 , -1.8970362 ]],

       [[-0.47082191, -0.47082191, -0.47082191, -0.47082191],
        [-0.47082191, -0.47082191, -0.47082191, -0.47082191],
        [-0.47082191, -0.47082191, -0.47082191, -0.47082191],
        [-0.47082191, -0.47082191, -0.47082191, -0.47082191]],

       [[ 0.60620538,  0.60620538,  0.60620538,  0.60620538],
        [ 0.60620538,  0.60620538,  0.60620538,  0.60620538],
        [ 0.60620538,  0.60620538,  0.60620538,  0.60620538],
        [ 0.60620538,  0.60620538,  0.60620538,  0.60620538]],

       [[ 1.07035183,  1.07035183,  1.07035183,  1.07035183],
        [ 1.07035183,  1.07035183,  1.07035183,  1.07035183],
        [ 1.07035183,  1.07035183,  1.07035183,  1.07035183],
        [ 1.07035183,  1.07035183,  1.07035183,  1.07035183]],

       [[ 0.8079782 ,  0.8079782 ,  0.8079782 ,  0.8079782 ],
        [ 0.8079782 ,  0.8079782 ,  0.8079782 ,  0.8079782 ],
        [ 0.8079782 ,  0.8079782 ,  0.8079782 ,  0.8079782 ],
        [ 0.8079782 ,  0.8079782 ,  0.8079782 ,  0.8079782 ]],

       [[-0.1166773 , -0.1166773 , -0.1166773 , -0.1166773 ],
        [-0.1166773 , -0.1166773 , -0.1166773 , -0.1166773 ],
        [-0.1166773 , -0.1166773 , -0.1166773 , -0.1166773 ],
        [-0.1166773 , -0.1166773 , -0.1166773 , -0.1166773 ]]])
Coordinates:
  * time     (time) int64 48B 0 1 2 3 4 5
  * lat      (lat) float64 32B -30.0 -10.0 10.0 30.0
  * lon      (lon) float64 32B 140.0 141.0 142.0 143.0
Attributes:
    units:    m

## Pattern 2 — global fit, patch-wise inference

A fitted sklearn estimator expects the same feature count during inference that it saw during fitting. For global-fit / patch-infer workflows, patch along the sample axis and keep the feature axis intact. Here the grid is stacked to `point × time`, PCA is fit globally over all points, and inference runs over point patches that all keep the same `time` features.

In [5]:
point_series = ssh.stack(point=("lat", "lon")).transpose("point", "time")
global_pca = XarrayEstimator(PCA(n_components=2), sample_dim="point").fit(point_series)

point_slices = [slice(0, 8), slice(8, 16)]
score_patches = [
    global_pca.transform(point_series.isel(point=slc)) for slc in point_slices
]
global_scores = xr.concat(score_patches, dim="point").sortby("point")
global_scores

<xarray.DataArray 'ssh' (point: 16, component: 2)> Size: 256B
array([[-7.38521157e-01,  0.00000000e+00],
       [-7.36071668e-01,  0.00000000e+00],
       [-7.33622178e-01,  0.00000000e+00],
       [-7.31172688e-01,  1.11022302e-16],
       [-2.48623209e-01,  1.11022302e-16],
       [-2.46173719e-01,  1.11022302e-16],
       [-2.43724229e-01, -1.11022302e-16],
       [-2.41274740e-01,  0.00000000e+00],
       [ 2.41274740e-01,  0.00000000e+00],
       [ 2.43724229e-01,  0.00000000e+00],
       [ 2.46173719e-01,  0.00000000e+00],
       [ 2.48623209e-01,  0.00000000e+00],
       [ 7.31172688e-01,  2.22044605e-16],
       [ 7.33622178e-01,  1.11022302e-16],
       [ 7.36071668e-01,  1.11022302e-16],
       [ 7.38521157e-01, -1.11022302e-16]])
Coordinates:
  * point      (point) object 128B (-30.0, 140.0) ... (30.0, 143.0)
  * component  (component) int64 16B 0 1
Attributes:
    units:    m

The equivalent `XRDAPatcher` recipe patches the stacked sample dimension and reconstructs the component scores:

```python
from xrpatcher import XRDAPatcher

point_series = ssh.stack(point=("lat", "lon")).transpose("point", "time")
pca = XarrayEstimator(PCA(n_components=10), sample_dim="point").fit(point_series)
patcher = XRDAPatcher(da=point_series, patches={"point": 4096}, strides={"point": 4096})
score_patches = [pca.transform(patcher[i]) for i in range(len(patcher))]
scores = patcher.reconstruct(score_patches)
```

The important invariant is that each patch still has the fitted feature axis (`time` here). No new xr_toolz operator is needed because `XRDAPatcher` yields `DataArray` patches and `XarrayEstimator` consumes `DataArray` inputs directly.

## Where this lands once OB-1.1 ships

The patch loops above use hand-written `isel` slices to keep this notebook executable without extra dependencies. The actual [`XRDAPatcher`](https://github.com/jejjohnson/xrpatcher) integration (operator wrappers, overlap-blended `reconstruct`, runtime caching) is tracked under issue OB-1.1 (#151). After that lands, the loops below shorten to:

```python
from xr_toolz.patcher import PatchDataset, PatchedInference

Sequential([
    PatchDataset(patches={"lat": 64, "lon": 64}, var="ssh"),
    PatchedInference(model=lambda da: scaler.fit_transform(da)),
])(ds)
```

Until then, the manual `isel` patterns shown here capture the same compositional shape on `xr.DataArray` only — no patcher dependency.
